In [ ]:
import pandas as pd
import numpy as np
import re

import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px

from warnings import filterwarnings

from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time

In [ ]:
df = pd.read_csv('weatherAUS.csv')

# Quick dataset overview

 - `Date` - Date of the record
 - `Location` - City name
 - `MinTemp` - The minimum temperature during a particular day `[Celsius degrees]`
 - `MaxTemp` - The maximum temperature during a particular day `[Celsius degrees]`
 - `Lluvia` - Rain during a particular day `[millimeters]`
 - `Evaporación` - Evaporation during a particular day `[millimeters]`
 - `Sunshine` - Bright sun during a particular day `[hours]`
 - `WindGusDir` - The direction of the strongest wind gust during a particular day `[16 points of the compass]`
 - `WindGustSpeed` ​​​​- Speed of the strongest wind gust during a particular day `[kilometers per hour]`
 - `WindDir9am` - Wind direction during the 10 previous minutes to 9 am `[16 points of the compass]`
 - `WindDir3pm` - Wind direction during the 10 previous minutes to 3:00 pm `[16 points of the compass]`
 - `WindSpeed ​​9am` - Wind speed during the 10 previous minutes to 9 am `[kilometers per hour]`
 - `WindSpeed3pm` - Wind speed during the 10 previous minutes to 3 pm `[kilometers per hour]`
 - `Humidity9am` - Wind humidity at 9 am `[percetage]`
 - `Humidity3pm` - Wind humidity at 3 pm `[percetage]`
 - `Presión9am` - Atmospheric Pressure at 9 am `[hectopascles]`
 - `Presión3pm` - Atmospheric Pressure at 3 pm `[hectopascles]`
 - `Cloud9am` - portion of the sky obscured by clouds at 9 am `[eighths of sky]`
 - `Cloud3pm` - portion of the sky obscured by clouds at 3 pm `[eighths of sky]`
 - `Temp9am` - Temperature at 9am `[Celsius degrees]`
 - `Temp3pm` - Temperature at 3 pm `[Celsius degrees]`
 - `RainToday` - If it rains today, then value is 1 `[yes]`. If it does not rain today, then value is 0 `[no]`
 - `RainTomorrow` - If it rains tomorrow, then value is 1 `[yes]`. If it does not rain tomorrow, then value is 0 `[no]`

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe().round(1)

> Let's see the amount of null values as a percentage of total rows

In [ ]:
round(df.isna().sum()/df.shape[0]*100, 2).sort_values(ascending=False)

> Some columns (top 4 above - all of them float64 type) have a lot of missing values.   
> Probably is not worthy (or too difficult) to complete this values without introducing too much bias to this columns

In [ ]:
df['Date'] = pd.DatetimeIndex(df['Date'])
df['Week'] = df['Date'].dt.isocalendar().week

In [ ]:
df.dtypes.value_counts()

> Since we have only few column types, we can easily split the analisis between categorical ('object') and numerical columns ('float64')

In [ ]:
df['Location'].nunique()

> The dataset has 49 different cities.  
> I will reduce this number by obtaining the state where those cities are: in the `State` column that will be created in the `df_location` dataframe and then merged into the main dataframe `df`

In [ ]:
df_location = pd.DataFrame(df['Location'].drop_duplicates())
df_location.reset_index(drop=True, inplace=True)
df_location.sort_values(by='Location', ascending=True, inplace=True)

In [ ]:
from utils import split_city_name
df_location['Location_split'] = df_location['Location'].apply(split_city_name)

In [ ]:
df_location['Latitude'] = None
df_location['Longitude'] = None
df_location['State'] = None

In [ ]:
geolocator = Nominatim(user_agent="AUS_geocoding (terciariala@gmail.com)")

In [ ]:
from utils import get_lat_lon
df_location[['Latitude', 'Longitude']] = df_location['Location_split'].apply(get_lat_lon).apply(pd.Series)

In [ ]:
df_location[['Latitude', 'Longitude']].isna().sum()

In [ ]:
from utils import get_state
df_location['State'] = df_location.apply(lambda row: get_state(row['Latitude'], row['Longitude'], row['Location_split']), axis=1)

In [ ]:
df = pd.merge(df, df_location, on='Location', how='left')

In [ ]:
df['State'].value_counts(sort=True, ascending=False)

> From 49 cities now we think in 9 states
> This project will be focused on the East coast therefore in the feature engineering script we will only keep the following states: `New South Wales`, `Victoria`, `Queensland` and `Australian Capital Territory`

> The pressure is indicated in Hectopascals so we will change the units to atmospheric units
> 1[ATM] = 1013.25 [HECTOPASCALS]

In [ ]:
from utils import pressure_to_hpa
df[['Pressure9am', 'Pressure3pm']] = df[['Pressure9am', 'Pressure3pm']].apply(pressure_to_hpa)

# EDA

## Univariate Analysis

### Categoric columns

In [ ]:
colours = ['#f2a73d', '#f76157', '#f6daab', '#dabd7b', '#f04155', '#ff823a', '#f2f26f', '#fff7bd', '#95cfb7', '#f40034',
           '#07f9a2', '#09c184', '#0a8967', '#0c5149', '#0d192b', '#fe1cac', '#820081', '#e4b302', '#e7204e', '#3f2c26']

In [ ]:
axtitle_dict = {'family': 'serif', 'color': 'darkred', 'weight': 'bold', 'size': 15}
axlab_dict = {'family': 'serif', 'color': 'black', 'size': 13}

In [ ]:
df = df.sort_values(by=['Location', 'Date'], axis=0, ascending=[True, True])

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.countplot(data=df, x=df['Date'].dt.year)
ax.set_title('Amount of records per year', fontdict=axtitle_dict)
ax.set_xlabel('Year', fontdict=axlab_dict)
plt.show()

> Amount of records by year is uneven so if we include records from '07, '08 and '17 we will probably enter bias to the model

In [ ]:
df.shape

In [ ]:
from utils import remove_records_by_year
df = remove_records_by_year(df, 'Date', [2007, 2008, 2017])
df.shape

In [ ]:
wind_directions = ['WindGustDir', 'WindDir9am', 'WindDir3pm']

In [ ]:
df[wind_directions].value_counts()

> Wind directions are classified into 16 different values.  
> In the Feature Engineering script I will reduce this to 8 according to the rose wind (see below)

![https://es.pinterest.com/pin/40673202879242300/](https://pin.it/4tqT2gol1)

In [ ]:
round(df[wind_directions].isna().sum()/df[wind_directions].shape[0]*100, 2).sort_values(ascending=False)

In [ ]:
filterwarnings("ignore", category=FutureWarning)
fig = plt.figure(figsize=[24, 8])

fig.suptitle('Wind directions - Count Plot', fontsize=18, fontweight = 'bold')
fig.subplots_adjust(top = 0.85, wspace = 0.25)

num_plots = len(wind_directions)
rows = (num_plots + 2) // 3

for i, wind in enumerate(wind_directions):
    non_null_values_wind_direction = df[wind].dropna() # variable created to skip null values in chart below

    if not non_null_values_wind_direction.empty:
        input = np.unique(non_null_values_wind_direction, return_counts=True)

        ax1 = fig.add_subplot(rows, 3, i+1)
        ax1 = sns.barplot(x=list(input[0]), y=list(input[1]))
        ax1.set_title(f'{wind}', fontdict=axtitle_dict)
    else:
        print(f"Column {wind} only contains nulls values, therefore is not included")

plt.show()

> - At 9am there is a clear predominance of northerly winds

> - In case of Gusts and 3pm wind, there isn't a clear predominance

In [ ]:
from utils import split_numeric_categorical_columns
numeric_columns, categorical_columns = split_numeric_categorical_columns(df)

In [ ]:
filterwarnings("ignore", category=FutureWarning)
fig = plt.figure(figsize=[10, 15])

fig.suptitle('Categorical columns (excluding wind directions) - Count plot', 
             fontsize=18, fontweight='bold')
fig.subplots_adjust(top=0.92, hspace=0.25, wspace=0.25)

for i, columns in enumerate(df[['State', 'RainToday', 'RainTomorrow']].columns):
    non_null_values = df[columns].dropna()

    if not non_null_values.empty:
        input = np.unique(non_null_values, return_counts=True)

        ax2 = fig.add_subplot(3, 1, i+1)
        ax2 = sns.barplot(x=list(input[0]), y=list(input[1]))
        ax2.set_title(f'{columns}', fontdict=axtitle_dict)
        ax2.tick_params(axis='x', rotation=90)
        plt.tight_layout(rect=[0, 0, 1, .98])
    else:
        print(f'Column {columns} only contains nulls values, therefore is not included')

> - Clearly for `RainToday` and `RainTomorrow` columns classes are imbalanced 

> - There are 3 locations with significantly fewer records than the others

### Numeric columns

In [ ]:
from utils import remove_columns_with_nulls
df, numeric_columns = remove_columns_with_nulls(df, numeric_columns, threshold=35)

In [ ]:
plt.figure(figsize=[15, 10])

cor = df.loc[:, numeric_columns].corr()

ax3 = sns.heatmap(cor, annot=True, fmt='.2f', cmap='coolwarm')
ax3.set_title('Correlation Heatmap (numeric columns)', 
              fontdict=axtitle_dict)

plt.show()

> - The majority of the `minimum temperatures` occurred at 9am
> - Same happens with `maximum temperature` at 3pm
> - In the feature selection phase before training the ML model, the high correlation among the `four temperature columns` suggests including ALL of them might be redundant and thus unnecessary.

> - `WindGustSpeed` do not seem to be related to the time: the correlation with wind speed is very similar with the records made at 9am and 3pm

In [ ]:
filterwarnings("ignore", category=FutureWarning)
fig = plt.figure(figsize=[24, 36])

fig.suptitle('Numerical fields - frequency distribution', 
             fontsize=18, fontweight = 'bold')
fig.subplots_adjust(top = 0.95, hspace = 0.5, wspace = 0.25)

num_plots = len(numeric_columns)
rows = (num_plots + 2) // 2

for i, columns in enumerate(numeric_columns):
    ax4 = fig.add_subplot(rows, 2, i+1)
    df[columns] = df[columns].replace([np.inf, -np.inf], np.nan)
    ax4 = sns.histplot(df[columns], color=colours[i])
    ax4.axvline(df[columns].quantile(q=0.25), color = 'green', 
                linestyle = '--')
    ax4.axvline(df[columns].mean(), color = 'red', 
                linestyle = '--')
    ax4.axvline(df[columns].median(), color = 'black', 
                linestyle = '--')
    ax4.axvline(df[columns].quantile(q=0.75), color = 'blue', 
                linestyle = '--')
    
    ax4.set_title(f'{columns}', fontdict=axtitle_dict)
    ax4.tick_params(labelsize=12)
    ax4.set_xlabel('')
    ax4.set_ylabel('')

    skewness = df[columns].skew()
    ax4.legend(labels=['25% Quartile', 'Mean', 'Median', '75% Quartile', 
                        f'Skewness: {skewness:.2f}'], 
                fontsize=10)

plt.show()

> - As we can see in the charts above, 8/12 numeric columns have a 'normal' distribution: mean and median are very close and skewness value in between -0.5 and +0.5

> - For the wind speed variables, a slight to moderate positive skew is observed: the mean is greater than the median, and skewness values range between 0 and 1.

> - In the `Rainfall` chart it is clear that there are a lot of outliers. An alternative conclusion would be that this is almost a binary variable: majority of days does not rain

> Let's see `Rainfall` distribution without the days with 0 precipitacion

In [ ]:
df_rainfall = df[df['Rainfall']>0]['Rainfall']

In [ ]:
filterwarnings("ignore", category=FutureWarning)

ax5 = sns.histplot(df_rainfall)
ax5.axvline(df_rainfall.quantile(q=0.25), color = 'green', 
            linestyle = '--')
ax5.axvline(df_rainfall.mean(), color = 'red', 
            linestyle = '--')
ax5.axvline(df_rainfall.median(), color = 'black', 
            linestyle = '--')
ax5.axvline(df_rainfall.quantile(q=0.75), color = 'blue', 
            linestyle = '--')

ax5.set_title('Rainfall frequency (excluding values <0)', 
              fontdict=axtitle_dict)
ax5.tick_params(labelsize=12)
ax5.set_xlabel('')
ax5.set_ylabel('')

skewness = df_rainfall.skew()
ax5.legend(labels=['25% Quartile', 'Mean', 'Median', '75% Quartile', 
                    f'Skewness: {skewness:.2f}'], 
            fontsize=10)
plt.show()

In [ ]:
df_rainfall.describe()

> Almost no changes to the analysis above
  
> `Rainfall` is like a binary TRUE or FALSE flag, similar to `RainToday` or `RainTomorrow`: Most days indicate that it did NOT rain, which is logical. Therefore, it would be interesting to know what happened on the day following those when it DID rain.

#### REVISAR

Conclusions:

1. In Australia the most common wind is from the West, from the Indian Ocean. This wind is called `Monzon` and generates strong tropical Rainfall in the summer, while dry and cold weather in the winter. This generates that the state called 'Western Australia' (15% of the dataset) has 2 opposite weather conditions throughout the year.

2. This makes particular sense when you consider that in the morning, the predominant winds are from the north, and in the afternoon, they come from the southeast. The winds shift across the western front during the 9 am to 15pm hour time span.

In [ ]:
df_weekly_rainfall = df.dropna(subset=['Rainfall']).groupby('Week')['Rainfall'].mean()

In [ ]:
plt.figure(figsize=(12,8))
plt.plot(df_weekly_rainfall.index, df_weekly_rainfall.values, marker='o', linestyle='--')
plt.xlabel('Week', fontdict=axlab_dict)
plt.ylabel('Weekly Rainfall [mm]', fontdict=axlab_dict)
plt.title('Mean Rainfall by Week', fontdict=axtitle_dict)
plt.grid(True)
plt.tight_layout()
plt.show()

#### meter algun comment aca

In [ ]:
plt.figure(figsize=(12, 8))

sns.boxplot(data=df, x='State', y='WindGustSpeed')

plt.xlabel('')
plt.ylabel('Wind Gust Speed [km/h]', fontdict=axlab_dict)
plt.title('Distribution of Wind Gust Speed by State', fontdict=axtitle_dict)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### meter algun comment aca

In [ ]:
columns_to_remove = ['Week','Rainfall', 'Latitude', 'Longitude', 'WindGustSpeed']

filtered_columns = []
for col in numeric_columns:
    if col not in columns_to_remove:
        filtered_columns.append(col)

In [ ]:
filterwarnings("ignore", category=FutureWarning)
fig = plt.figure(figsize=[24, 16])

fig.suptitle('Boxplots', fontsize=18, fontweight = 'bold')
fig.subplots_adjust(top=0.92, hspace=0.5, wspace=0.25)

num_plots = len(filtered_columns)
rows = (num_plots + 2) // 4

for i, col in enumerate(filtered_columns):
  ax6 = fig.add_subplot(rows, 4, i+1)
  ax6 = sns.boxplot(data = df, x=col, color= colours[i])
  ax6.set_title(f'{col}', fontdict=axtitle_dict)
  ax6.set_xlabel(f'{col}', fontdict=axlab_dict)

plt.show()

## Bivariate Analysis

### Categoric columns

In [ ]:
import folium
from folium.plugins import HeatMap

In [ ]:
center_lat = df_location['Latitude'].astype(float).mean()
center_lon = df_location['Longitude'].astype(float).mean()

folium_hmap = folium.Map(location=[center_lat, center_lon], 
                         zoom_start=3.5, tiles='OpenStreetMap')

weight = df['Location'].value_counts()
hm_wide = HeatMap(data=list(zip(df_location['Latitude'].astype(float), 
                                df_location['Longitude'].astype(float), 
                                df_location['Location'].map(weight))),
                  min_opacity=.8, radius=15, blur=15, max_zoom=10)

folium_hmap.add_child(hm_wide)

> Clearly the majority of cities included are in South East coast

In [ ]:
fig = plt.figure(figsize=(24, 10))
fig.suptitle('Frequency of wind directions', fontsize=18, 
             fontweight='bold')
fig.subplots_adjust(top=0.92, wspace=0.25)

num_plots = len(wind_directions)
rows = (num_plots + 2) // 3

for i, wind_dir in enumerate(wind_directions):
    ax7 = fig.add_subplot(rows, 3, i+1)
    ax7 = sns.countplot(x = df[wind_dir], 
                        hue = df['RainTomorrow'])
    ax7.set_xlabel('')
    ax7.set_ylabel('')
    ax7.set_title(f'{wind_dir}', fontdict=axtitle_dict)
    ax7.grid(visible=True, axis='y', which='major')
    
plt.show()

> There's a highly variable relationship between the incidence of the direction from which the wind blows and whether it ends up raining or not.

> Despite having a comparable number of samples in the dataset (as seen in univariate analysis), the frequency of rainfall is about 50% higher when the wind originates from the SSW direction compared to the ENE:
> - `ENE`: [yes≈1.200, no≈6.500]
> - `SSW`: [yes≈2.000, no≈6.000]

### Numeric columns

In [ ]:
filterwarnings('ignore', category=FutureWarning)
fig = plt.figure(figsize=(16,9))
ax8 = sns.scatterplot(data=df, x='Humidity9am', y='Humidity3pm',
                      hue='RainTomorrow')
ax8.set_title("Pressures during day as indicator of tomorrow's rain", fontdict=axtitle_dict)
ax8.set_xlabel('Humidity at 9am [%]', fontdict=axlab_dict)
ax8.set_ylabel('Humidity at 3pm [%]', fontdict=axlab_dict)
plt.show()

> - A combination of `Humidity` levels above 50% at 9am and above 80% at 3pm suggest a high likelihood of rain on the following day.

> - Afternoon `Humidity` levels (at 3pm) appear to be a better predictor of rain for the next day.

In [ ]:
filterwarnings('ignore', category=FutureWarning)

fig = plt.figure(figsize=(16,9))

ax9 = sns.scatterplot(data=df, x='Pressure9am', y='Pressure3pm',
                      hue='RainTomorrow')
ax9.set_title("Pressures during day as indicator of tomorrow's rain", fontdict=axtitle_dict)
ax9.set_xlabel('Pressure at 9am [atm]', fontdict=axlab_dict)
ax9.set_ylabel('Pressure at 3pm [atm]', fontdict=axlab_dict)
plt.show()

> - A combination of `Pressure` levels below 1[ATM] both at 9am and 3pm suggest a high likelihood of rain on the following day.

In [ ]:
df['TempRange'] = df['MaxTemp'] - df['MinTemp']

In [ ]:
filterwarnings('ignore', category=FutureWarning)

fig = plt.figure(figsize=(16,9))

ax10 = sns.scatterplot(data=df, x='MinTemp', y='MaxTemp',
                      hue='RainTomorrow', size='TempRange', 
                      sizes=(100, 1), alpha=0.8)
ax10.set_title("Temperatures as indicator of tomorrow's rain", fontdict=axtitle_dict)
ax10.set_xlabel('MinTemp [celsius degrees]', fontdict=axlab_dict)
ax10.set_ylabel('MaxTemp [celsius degrees]', fontdict=axlab_dict)
plt.show()

> Lower `Temperatures` AND lower `Temperature Range` suggest a high likelihood of rain on the following day.

In [ ]:
fig = plt.figure(figsize=(10,7))
ax11 = sns.heatmap(pd.crosstab(df['RainToday'], df['RainTomorrow']), 
                  annot=True, fmt='.0f', cmap='coolwarm')
ax11.set_title("Today's rain, tomorrow's forecast?", fontdict=axtitle_dict)
ax11.set_xlabel('Rain Today', fontdict=axlab_dict)
ax11.set_ylabel('Rain Tomorrow', fontdict=axlab_dict)
plt.show()

> Conclusions:
> 1. Higher `Humidity` levels (espcially at 3pm) with low level `Pressures` and low `temperatures` during the day before suggest a high likelihood of rain on the following day.
> 2. Knowing whether it `RainToday` does not provide a trustworthy basis for predicting `RainTomorrow`.

> Let's verify that first preliminary conclusion

In [ ]:
fig, ax12 = plt.subplots(1, 3, figsize=(20, 6))

# Scatter plot 1: 'Humidity9am' vs 'Humidity3pm'
sns.scatterplot(data=df, x='Humidity9am', y='Humidity3pm', 
                hue='RainTomorrow', ax=ax12[0], 
                palette={'Yes': 'red', 'No': 'blue'})
ax12[0].set_title('Humidity 9am vs 3pm')
ax12[0].axvline(50, color='black', 
                linestyle='--') # Line for 50% humidity at 9am
ax12[0].axhline(80, color='black', 
                linestyle='--') # Line for 80% humidity at 3pm

# Scatter plot 2: 'Pressure9am' vs 'Pressure3pm'
sns.scatterplot(data=df, x='Pressure9am', y='Pressure3pm', 
                hue='RainTomorrow', ax=ax12[1], 
                palette={'Yes': 'red', 'No': 'blue'})
ax12[1].set_title('Pressure 9am vs 3pm')
ax12[1].axvline(1, color='black', 
                linestyle='--') # Line for pressure 9am = 1atm
ax12[1].axhline(1, color='black', 
                linestyle='--') # Line for pressure 3pam = 1atm

# Scatter plot 3: 'MinTemp' vs 'MaxTemp'
sns.scatterplot(data=df, x='MinTemp', y='MaxTemp', hue='RainTomorrow', 
                ax=ax12[2], palette={'Yes': 'red', 'No': 'blue'})
ax12[2].set_title('MinTemp vs MaxTemp')

plt.tight_layout()
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
df['RainTomorrow_numeric'] = df['RainTomorrow'].map({'No': 0, 'Yes': 1})

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax13 = fig.add_subplot(111, projection='3d')

ax13.scatter(
    df['Humidity3pm'], df['Pressure3pm'], df['TempRange'], 
    c=df['RainTomorrow_numeric'], alpha=0.5)

ax13.set_xlabel("Humidity at 3PM")
ax13.set_ylabel("Pressure at 3PM")
ax13.set_zlabel("Temperature Range")
ax13.set_title("3D Scatter Plot of Weather Variables and RainTomorrow")

plt.show()

In [ ]:
fig = px.scatter_3d(df, x='Humidity3pm', y='Pressure3pm', z='TempRange', 
                     color='RainTomorrow_numeric', opacity=0.7)
fig.update_traces(marker=dict(size=5))
fig.update_layout(title="3D Scatter Plot of Weather Variables and RainTomorrow")
fig.show()